# Web Hosting

This example recreates the first CloudLab example where we launch a web server. I will also provide the example to enable 
SSH tunneling to access this server from the local computer. 

---

## Why Use SSH Tunnels?

Many FABRIC VMs are not directly reachable from the public Internet. Instead, access is typically provided through a **bastion host**. SSH tunnels allow you to forward ports securely through the bastion to reach internal services running on private VMs.

## Import the FABlib Library

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.get_image_names()

## Create the Experiment Slice



In [ ]:
slice_name = "web_hosting_imperative"

In [ ]:
# Create a slice
slice = fablib.new_slice(name=slice_name)

# Add a node
node = slice.add_node(name="Node1", disk=10, image='default_ubuntu_22')

# Submit the slice
slice.submit();

In [ ]:
slice.update()

print("Slice state:", slice.get_state())
print("Slice stable:", slice.isStable())

for node in slice.get_nodes():
    print("----", node.get_name(), "----")
    print("reservation state:", node.get_reservation_state())
    print("management ip:", node.get_management_ip())
    print("username:", node.get_username())
    print("error:", node.get_error_message())
    print(node.get_ssh_command())

## Run a Web Service on the Node 

Test that the web service is up on the node slice

In [ ]:
slice = fablib.get_slice(slice_name)
node = slice.get_node('Node1')


node.upload_directory('scripts','.')
node.execute('bash scripts/web_server.sh',  quiet=True, output_file=f"{node.get_name()}.log")

In [ ]:
node.execute("curl localhost:80", quiet=True, output_file=f"{node.get_name()}.log");

## Start the SSH Tunnel

- Create SSH Tunnel Configuration `fabric_ssh_tunnel_tools.zip`
- Download your custom `fabric_ssh_tunnel_tools.zip` tarball from the `fabric_config` folder.  
- Untar the tarball and put the resulting folder (`fabric_ssh_tunnel_tools`) somewhere you can access it from the command line.
- Open a terminal window. (Windows: use `powershell`) 
- Use `cd` to navigate to the `fabric_ssh_tunnel_tools` folder.
- Run the following command to setup permission correctly

```bash
chmod 600 slice_key fabric-bastion-key
```

- In your terminal, run the command that results from running the following cell (leave the terminal window open).

In [ ]:
fablib.create_ssh_tunnel_config(overwrite=True)

#### Port Forwarding Example

FileBrowser is running on the VM at port `5555`, use the following SSH command from your laptop:

In [ ]:
import os
# Port on your local machine that you want to map the web server to. This should be a port that you have specified on 
# docker-compose.yml

local_port='5555'
# We use 0.0.0.0 because we want the ability to forward this interface outside of the container. 
local_host='0.0.0.0'

# Port on the node used by the web server
target_port='80'

# Username/node on FABRIC
target_host=f'{node.get_username()}@{node.get_management_ip()}'

print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')

## Connect to the Web Server from your local browser

The browser service on the node is now mapped to 127.0.0.1:5555 on your local machine. You can open a browser and navigate to the following address (or just click the link): 

[http://127.0.0.1:5555](http://127.0.0.1:5555)

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice.delete()